# Classificação do tipo de página

Este notebook compara a resposta de `requests` com sinais comuns de páginas estáticas, SPA e páginas dinâmicas.

A ideia é responder, para cada URL:
- o HTML já vem completo no servidor?
- a página é apenas um esqueleto com JavaScript?
- o conteúdo depende de interação com filtros, botões ou carregamento posterior?

In [1]:
import json
import re
from pprint import pprint

import requests

URLS = [
    "https://www.inca.gov.br/MortalidadeWeb/pages/Modelo10/consultar.xhtml",
    "https://irhc.inca.gov.br/RHCNet/visualizaTabNetExterno.action",
    "https://tabnet.datasus.gov.br/cgi/dhdat.exe?siscan/mamografia_atendmg.def",
]

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 "
        "(KHTML, like Gecko) Chrome/125.0.0.0 Safari/537.36"
    )
}

In [3]:
def _strip_scripts(html: str) -> str:
    html = re.sub(r"<script\b[^>]*>.*?</script>", "", html, flags=re.I | re.S)
    html = re.sub(r"<style\b[^>]*>.*?</style>", "", html, flags=re.I | re.S)
    html = re.sub(r"<[^>]+>", " ", html)
    html = re.sub(r"\s+", " ", html).strip()
    return html


def classificar_pagina(url: str) -> dict:
    try:
        resposta = requests.get(url, headers=HEADERS, timeout=(10, 90))
        resposta.raise_for_status()
        html = resposta.text or ""
        lower = html.lower()

        texto_visivel = _strip_scripts(html)
        tamanho_html = len(html)
        tamanho_texto = len(texto_visivel)
        proporcao_texto = (tamanho_texto / tamanho_html) if tamanho_html else 0

        marcadores_js = []
        if "__next_data__" in lower or "_next/static" in lower:
            marcadores_js.append("Next.js")
        if "ng-version" in lower:
            marcadores_js.append("Angular")
        if "data-reactroot" in lower or ("react" in lower and "root" in lower):
            marcadores_js.append("React")
        if re.search(r"id=['\"]root['\"]", lower) or re.search(r"id=['\"]app['\"]", lower):
            marcadores_js.append("shell/app root")

        tem_formulario = "<form" in lower or "<select" in lower or "<input" in lower
        tem_tabela = "<table" in lower
        tem_muito_script = lower.count("<script") >= 5

        if (
            tamanho_html < 5000
            and (marcadores_js or not texto_visivel)
            and tem_muito_script
        ):
            tipo = "SPA ou aplicação renderizada no navegador"
        elif tem_formulario or tem_tabela:
            tipo = "página dinâmica ou server-rendered interativa"
        else:
            tipo = "página estática ou server-rendered"

        return {
            "url": url,
            "status_code": resposta.status_code,
            "content_type": resposta.headers.get("content-type", ""),
            "html_len": tamanho_html,
            "visible_text_len": tamanho_texto,
            "visible_text_ratio": round(proporcao_texto, 4),
            "script_tags": lower.count("<script"),
            "form_tags": lower.count("<form"),
            "select_tags": lower.count("<select"),
            "table_tags": lower.count("<table"),
            "js_markers": marcadores_js,
            "diagnostico": tipo,
            "erro": None,
        }

    except requests.exceptions.RequestException as exc:
        return {
            "url": url,
            "status_code": None,
            "content_type": None,
            "html_len": 0,
            "visible_text_len": 0,
            "visible_text_ratio": 0,
            "script_tags": 0,
            "form_tags": 0,
            "select_tags": 0,
            "table_tags": 0,
            "js_markers": [],
            "diagnostico": "falha na requisição via requests",
            "erro": str(exc),
        }


resultados = [classificar_pagina(url) for url in URLS]
pprint(resultados)

[{'content_type': 'text/html;charset=UTF-8',
  'diagnostico': 'página dinâmica ou server-rendered interativa',
  'erro': None,
  'form_tags': 7,
  'html_len': 80576,
  'js_markers': [],
  'script_tags': 57,
  'select_tags': 12,
  'status_code': 200,
  'table_tags': 28,
  'url': 'https://www.inca.gov.br/MortalidadeWeb/pages/Modelo10/consultar.xhtml',
  'visible_text_len': 6405,
  'visible_text_ratio': 0.0795},
 {'content_type': 'text/html;charset=UTF-8',
  'diagnostico': 'página dinâmica ou server-rendered interativa',
  'erro': None,
  'form_tags': 2,
  'html_len': 14002,
  'js_markers': [],
  'script_tags': 3,
  'select_tags': 0,
  'status_code': 200,
  'table_tags': 10,
  'url': 'https://irhc.inca.gov.br/RHCNet/visualizaTabNetExterno.action',
  'visible_text_len': 3491,
  'visible_text_ratio': 0.2493},
 {'content_type': 'text/html',
  'diagnostico': 'página dinâmica ou server-rendered interativa',
  'erro': None,
  'form_tags': 1,
  'html_len': 986153,
  'js_markers': [],
  'script_t

In [4]:
for item in resultados:
    print("\nURL:", item["url"])
    print("Status:", item["status_code"])
    print("Content-Type:", item["content_type"])
    print("HTML:", item["html_len"], "chars")
    print("Texto visível:", item["visible_text_len"], "chars")
    print("Marcadores JS:", item["js_markers"])
    print("Diagnóstico:", item["diagnostico"])


URL: https://www.inca.gov.br/MortalidadeWeb/pages/Modelo10/consultar.xhtml
Status: 200
Content-Type: text/html;charset=UTF-8
HTML: 80576 chars
Texto visível: 6405 chars
Marcadores JS: []
Diagnóstico: página dinâmica ou server-rendered interativa

URL: https://irhc.inca.gov.br/RHCNet/visualizaTabNetExterno.action
Status: 200
Content-Type: text/html;charset=UTF-8
HTML: 14002 chars
Texto visível: 3491 chars
Marcadores JS: []
Diagnóstico: página dinâmica ou server-rendered interativa

URL: https://tabnet.datasus.gov.br/cgi/dhdat.exe?siscan/mamografia_atendmg.def
Status: 200
Content-Type: text/html
HTML: 986153 chars
Texto visível: 336242 chars
Marcadores JS: []
Diagnóstico: página dinâmica ou server-rendered interativa


## Como interpretar

- Se `requests` já entrega o conteúdo completo, a página tende a ser estática ou server-rendered.
- Se a resposta vier quase vazia e o navegador mostrar conteúdo após JavaScript, a página tende a ser SPA.
- Se a página exige cliques, filtros, selects, abas ou download gerado depois da interação, ela é dinâmica.

Para uma confirmação definitiva, compare o HTML de `requests` com o que aparece no navegador.